# Day D3 — 공간 정규화, 템플릿, 아틀라스: 서로 다른 뇌를 한 좌표에 놓기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/D-fmri/day-D3.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/D-fmri/day-D3.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 보간을 거듭하면 영상이 번진다, 라벨 영상은 nearest로

In [ ]:
import numpy as np
from scipy import ndimage

# 가상 '구조 영상' 한 장: 줄무늬가 있는 원 (세밀한 경계가 많은 그림)
yy, xx = np.mgrid[0:96, 0:96]
inside = (xx - 48) ** 2 + (yy - 48) ** 2 < 40 ** 2
img = ((xx // 6) % 2).astype(float) * inside

def roundtrip(order, n_steps):
    """30도 돌렸다가 다시 30도 되돌린다. n_steps번에 나눠 돌리면 그만큼 보간을 더 한다."""
    out = img.copy()
    for angle in [30 / n_steps] * n_steps + [-30 / n_steps] * n_steps:
        out = ndimage.rotate(out, angle, reshape=False, order=order)
    return np.sqrt(np.mean((out - img)[inside] ** 2))   # 원본과의 오차(RMSE)

print("보간 방식       2번 보간(합친 변환)  12번 보간(단계마다 저장)")
for order, name in [(0, "nearest"), (1, "linear"), (3, "cubic spline")]:
    print(f"{name:13} {roundtrip(order, 1):18.3f} {roundtrip(order, 6):20.3f}")

# 라벨 영상(아틀라스)은 값이 '이름표'다. 보간 방식에 따라 없는 라벨이 생긴다
labels = np.zeros((40, 40)); labels[5:20, 5:35] = 12; labels[20:35, 5:35] = 20   # 가상 구조 12번, 20번
for order in [0, 1]:
    moved = np.round(ndimage.rotate(labels, 10, reshape=False, order=order), 1)
    print("order=%d 뒤 라벨 값 종류: %d개, 예: %s" % (order, np.unique(moved).size, np.unique(moved)[:6]))

### 예시 2 — 대비가 뒤집힌 두 영상: 상관과 상호정보량

In [ ]:
import numpy as np

rng = np.random.default_rng(5)
yy, xx = np.mgrid[0:80, 0:80]
r = np.sqrt((xx - 40) ** 2 + (yy - 40) ** 2)
tissue = np.select([r < 12, r < 25, r < 34], [1, 2, 3], 0)   # 0 바깥, 1 '뇌실', 2 '백질', 3 '회백질'

# 같은 조직이 영상 종류에 따라 밝기 순서가 다르다 (가상 대비)
t1_like = np.choose(tissue, [0, 20, 100, 60]) + rng.normal(0, 5, tissue.shape)
t2_like = np.choose(tissue, [0, 100, 30, 60]) + rng.normal(0, 5, tissue.shape)

def mutual_info(a, b, bins=32):
    p, _, _ = np.histogram2d(a.ravel(), b.ravel(), bins=bins)
    p = p / p.sum()
    pa, pb = p.sum(1, keepdims=True), p.sum(0, keepdims=True)
    nz = p > 0
    return float((p[nz] * np.log(p[nz] / (pa @ pb)[nz])).sum())

head = r < 34                                   # 머리 안쪽만 비교한다
print("이동(voxel)  상관계수   상호정보량(MI)")
for shift in [0, 2, 5, 10]:
    moved = np.roll(t2_like, shift, axis=1)
    a, b = t1_like[head], moved[head]
    print(f"{shift:>8}   {np.corrcoef(a, b)[0, 1]:8.3f}   {mutual_info(a, b):10.3f}")